## Assignment 1
### Task 1.1 Calculate the Clock Granularity of different Python Timers
We start with defining the given "checktick"-function to calculate the clock granularity of the "time.time()" clock.

In [ ]:
from timeit import default_timer as timer
import numpy as np
import time

def checktick():
    M = 200
    timesfound = np.empty((M,))
    for i in range(M):
        t1 =  time.time() # get timestamp from timer
        t2 = time.time() # get timestamp from timer
        while (t2 - t1) < 1e-16: # if zero then we are below clock granularity, retake timing
            t2 = time.time() # get timestamp from timer
        t1 = t2 # this is outside the loop
        timesfound[i] = t1 # record the time stamp
    minDelta = 1000000
    Delta = np.diff(timesfound) # it should be cast to int only when needed
    minDelta = Delta.min() * 1e9 # convert to nanoseconds
    return minDelta
print("Clock granularity of time.time() is ", checktick(), " nanoseconds")

Clock granularity is  715.2557373046875  nanoseconds


Now we modify the "checktick"-function to calculate the clock granularity of "timeit".

In [ ]:
def checktick():
    M = 200
    timesfound = np.empty((M,))
    for i in range(M):
        t1 =  timer() # get timestamp from timer
        t2 = timer() # get timestamp from timer
        while (t2 - t1) < 1e-16: # if zero then we are below clock granularity, retake timing
            t2 = timer() # get timestamp from timer
        t1 = t2 # this is outside the loop
        timesfound[i] = t1 # record the time stamp
    minDelta = 1000000
    Delta = np.diff(timesfound) # it should be cast to int only when needed
    minDelta = Delta.min() * 1e9 # convert to nanoseconds
    return minDelta
print("Clock granularity of timeit is ", checktick(), " nanoseconds")

Clock granularity of timeit is  166.00097296759486  nanoseconds


Now we modify it for the "time.time_ns" timer:

In [9]:
def checktick():
    M = 200
    timesfound = np.empty((M,))
    for i in range(M):
        t1 =  time.time_ns() # get timestamp from timer
        t2 = time.time_ns() # get timestamp from timer
        while (t2 - t1) < 1e-16: # if zero then we are below clock granularity, retake timing
            t2 = time.time_ns() # get timestamp from timer
        t1 = t2 # this is outside the loop
        timesfound[i] = t1 # record the time stamp
    minDelta = 1000000
    Delta = np.diff(timesfound) # it should be cast to int only when needed
    minDelta = Delta.min()
    return minDelta
print("Clock granularity of time.time_ns is ", checktick(), " nanoseconds")

Clock granularity of time.time_ns is  768.0  nanoseconds


The results show, that the "timeit" timer has the smallest granularity.

### Task 1.2 Timing the Julia set code functions.

We modified the given decorator and put it in a new file, to keep the original one. The new code is in the file namend "JuliaSet-timing".
Changes made to the decorator:
- Change the timer to the one from timeit, because it turned out to have the smallest clock granularity.
- Run Code multiple times (5 times), to get average and standard deviation data.

Modified code of the decorator:
```python
def timefn(fn):
    @wraps(fn)
    def measure_time(*args, **kwargs):
        n = 5
        times = []

        for i in range(n):
            t1 = timer()
            result = fn(*args, **kwargs)
            t2 = timer()
            times.append(t2 - t1)
        avg = np.mean(times)
        std = np.std(times)
        print(f"@timefn: {fn.__name__} took {avg:.6f} seconds, standard deviation: {std:.6f} (n={n})")
        return result
    return measure_time
```
Since the "calc_pure_python" function calls the "calculate_z_serial_purepython" inside itself, we first only timed the "calc_pure_python" function (in JuliaSet-timing.py).

In [12]:
%run JuliaSet-timing.py

Length of x: 1000
Total elements: 1000000
Length of x: 1000
Total elements: 1000000
Length of x: 1000
Total elements: 1000000
Length of x: 1000
Total elements: 1000000
Length of x: 1000
Total elements: 1000000
@timefn: calc_pure_python took 3.910685 seconds, standard deviation: 0.024349 (n=5)


Now the function "calculate_z_serial_purepython" is timed, for that we use the exact same code as before but only with the decorator used on "calculate_z_serial_purepython".
This is done in a separate file "JuliaSet-timing2.py".

In [14]:
%run JuliaSet-timing2.py

Length of x: 1000
Total elements: 1000000
@timefn: calculate_z_serial_purepython took 3.642976 seconds, standard deviation: 0.029518 (n=5)


Since the standard deviation is far bigger than the clock granularity, the timer function is not limited by the clock granularity. If it was the same value as the clock granularity, this would mean that the timer is not fast enough and the results depend on the timer itself.

### Task 1.3 Profile the Julia set code with cProfile and line_profiler the computation.

Since "cProfile" doesn't need changes in the code we can use the original code in the file "JuliaSet-original.py". We will time the execution with the profilers to analyze the overhead afterwards.

In [17]:
t1 =  timer()
!python -m cProfile -s cumulative JuliaSet-original.py
t2 = timer()
time_cProfiler = t2 - t1
print("Execution time with cProfile: ", time_cProfiler, " seconds")

Length of x: 1000
Total elements: 1000000
calculate_z_serial_purepython took 4.800048112869263 seconds
         36221995 function calls in 5.110 seconds

   Ordered by: cumulative time

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
        1    0.000    0.000    5.110    5.110 {built-in method builtins.exec}
        1    0.012    0.012    5.109    5.109 JuliaSet-original.py:1(<module>)
        1    0.246    0.246    5.098    5.098 JuliaSet-original.py:21(calc_pure_python)
        1    3.847    3.847    4.800    4.800 JuliaSet-original.py:59(calculate_z_serial_purepython)
 34219980    0.953    0.000    0.953    0.000 {built-in method builtins.abs}
  2002000    0.047    0.000    0.047    0.000 {method 'append' of 'list' objects}
        1    0.004    0.004    0.004    0.004 {built-in method builtins.sum}
        3    0.000    0.000    0.000    0.000 {built-in method builtins.print}
        2    0.000    0.000    0.000    0.000 {built-in method time.time}
       

Now let's save it to a file so we can visualize the results with snakeviz afterwards.

In [18]:
!python -m cProfile -o profile.stats JuliaSet-original.py

Length of x: 1000
Total elements: 1000000
calculate_z_serial_purepython took 4.916852951049805 seconds


In [ ]:
!python -m snakeviz profile.stats --server

snakeviz web server started on 127.0.0.1:8080; enter Ctrl-C to exit
http://127.0.0.1:8080/snakeviz/%2FUsers%2Ftimeberle%2FGitHub%2FHPC-Intro%2Fprofile.stats
^C


First we will use the lineProfiler for the first function "calc_pure_python". Therefore we need to use the decorator only on this function, which is done in the file "JuliaSet-lineProfiler.py".

In [20]:
t1 =  timer()
!python -m kernprof -l JuliaSet-lineProfiler.py
t2 =  timer()
!python -m line_profiler JuliaSet-lineProfiler.py.lprof
time_lineProfiler1 = t2 - t1
print("Execution time with lineProfiler on calc_pure_python: ", time_lineProfiler1, " seconds")

Length of x: 1000
Total elements: 1000000
calculate_z_serial_purepython took 14.485118389129639 seconds
Wrote profile results to 'JuliaSet-lineProfiler.py.lprof'
Inspect results with:
python -m line_profiler -rmt JuliaSet-lineProfiler.py.lprof
Timer unit: 1e-06 s

Total time: 15.1685 s
File: JuliaSet-lineProfiler.py
Function: calc_pure_python at line 20

Line #      Hits         Time  Per Hit   % Time  Line Contents
    20                                           @profile
    21                                           def calc_pure_python(desired_width, max_iterations):
    22                                               """Create a list of complex coordinates (zs) and complex parameters (cs),
    23                                               build Julia set"""
    24         1          1.0      1.0      0.0      x_step = (x2 - x1) / desired_width
    25         1          0.0      0.0      0.0      y_step = (y1 - y2) / desired_width
    26         1          1.0      1.0      0

Now let's do it also for the function "calculate_z_serial_purepython", in file "JuliaSet-lineProfiler2.py".

In [21]:
t1 =  timer()
!python -m kernprof -l JuliaSet-lineProfiler2.py
t2 =  timer()
!python -m line_profiler JuliaSet-lineProfiler2.py.lprof
time_lineProfiler2 = t2 - t1
print("Execution time with lineProfiler on calculate_z_serial_purepython: ", time_lineProfiler2, " seconds")

Length of x: 1000
Total elements: 1000000
calculate_z_serial_purepython took 30.517379999160767 seconds
Wrote profile results to 'JuliaSet-lineProfiler2.py.lprof'
Inspect results with:
python -m line_profiler -rmt JuliaSet-lineProfiler2.py.lprof
Timer unit: 1e-06 s

Total time: 18.9274 s
File: JuliaSet-lineProfiler2.py
Function: calculate_z_serial_purepython at line 58

Line #      Hits         Time  Per Hit   % Time  Line Contents
    58                                           @profile
    59                                           def calculate_z_serial_purepython(maxiter, zs, cs):
    60                                               """Calculate output list using Julia update rule"""
    61         1        684.0    684.0      0.0      output = [0] * len(zs)
    62   1000001     170707.0      0.2      0.9      for i in range(len(zs)):
    63   1000000     133098.0      0.1      0.7          n = 0
    64   1000000     163971.0      0.2      0.9          z = zs[i]
    65   1000000

To analyze the overhead of the profilers we will compare the time it takes to run the code with and without the profilers. Therefore we will run it once without the profilers.

In [22]:
t1 =  timer()
%run JuliaSet-original.py
t2 = timer()
time_noProfiler = t2 - t1
print("Execution time without profiler: ", time_noProfiler, " seconds")
overhead_cProfiler = time_cProfiler - time_noProfiler
overhead_lineProfiler1 = time_lineProfiler1 - time_noProfiler
overhead_lineProfiler2 = time_lineProfiler2 - time_noProfiler
print("Overhead of cProfile: ", overhead_cProfiler, " seconds")
print("Overhead of lineProfiler on calc_pure_python: ", overhead_lineProfiler1, " seconds")
print("Overhead of lineProfiler on calculate_z_serial_purepython: ", overhead_lineProfiler2, " seconds")

Length of x: 1000
Total elements: 1000000
calculate_z_serial_purepython took 3.651555061340332 seconds
Execution time without profiler:  3.903508833995147  seconds
Overhead of cProfile:  1.5332842910065665  seconds
Overhead of lineProfiler on calc_pure_python:  11.987149000007776  seconds
Overhead of lineProfiler on calculate_z_serial_purepython:  27.208218416002637  seconds


### Task 1.4 Memory-profile the Juliaset code. Use the memory_profiler and mprof to profile the computation in JuliaSet code.